In [7]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [ ]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [9]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [10]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [2.88148707e+02 1.07749860e+05 2.82064749e+02 1.00140029e+04
 4.82923578e+02 2.06931435e+03 8.97512400e+01]
Error: 31100.05
Simulating: [2.62637465e+02 4.33023215e+05 2.28339296e+03 4.62110139e+04
 2.80379130e+02 7.39289208e+03 3.29149972e+01]
Error: 34577.66
Simulating: [2.13163068e+02 4.87862115e+05 1.35027168e+03 4.18885121e+04
 4.20771051e+02 1.65844255e+02 7.22569302e+01]
Error: 49103.90
Simulating: [3.78830614e+02 3.80829084e+05 1.83494939e+03 2.00212650e+04
 4.59939220e+02 3.56068298e+03 9.11478148e+01]
Error: 41924.95
Simulating: [2.73717755e+02 1.17943941e+05 3.92634799e+03 1.31868162e+04
 2.30035290e+02 9.67753777e+03 4.37662056e+01]
Error: 12664.73
Simulating: [2.42442486e+02 4.65483619e+05 6.54344008e+02 8.78443468e+03
 7.43641145e+01 5.83626018e+03 7.44666673e+01]
Error: 386010.79
Simulating: [2.36072918e+02 2.12855650e+05 4.78572456e+03 1.70982974e+03
 5.29459455e+01 9.09239170e+03 2.85898026e+01]
Error: 21420.76
Simulating: [3.63766690e+02 4.82398558e+05 2.66